# 01  Exploratory Data Analysis

Using `data/sample/synthetic_receipts.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

pd.set_option('display.max_columns', None)

In [ ]:
file_path = "../data/sample/synthetic_receipts.csv"

df = pd.read_csv(file_path)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
# Convert timestamp to datetime and engineer some basic time features
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['dow'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour

df.head()

In [ ]:
# Basic checks: is line_total always quantity * unit_price?

calc_line_total = (df['quantity'] * df['unit_price']).round(2)

mismatch = (calc_line_total != df['line_total'].round(2))
print(f"Mismatched rows: {mismatch.sum()}")

if mismatch.any():
    display(df.loc[mismatch].head())

In [ ]:
# Sales volume and revenue by product

prod_summary = df.groupby('product_name').agg({
    'quantity': 'sum',
    'line_total': 'sum',
    'transaction_id': 'nunique'
}).rename(columns={'transaction_id': 'num_transactions'})

prod_summary.sort_values('line_total', ascending=False).head(10)

In [ ]:
plt.figure(figsize=(10, 5))
prod_summary.sort_values('line_total', ascending=False)['line_total'].head(15).plot(kind='bar')
plt.ylabel('Revenue')
plt.title('Top 15 Products by Revenue')
plt.tight_layout()

In [ ]:
# Daily revenue and quantity

daily = df.groupby('date').agg({
    'line_total': 'sum',
    'quantity': 'sum'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(daily['date'], daily['line_total'], marker='o', label='Revenue')
ax1.set_ylabel('Revenue')
ax1.set_title('Daily Revenue Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

fig, ax2 = plt.subplots(figsize=(12, 5))
ax2.plot(daily['date'], daily['quantity'], marker='o', color='orange', label='Quantity')
ax2.set_ylabel('Quantity')
ax2.set_title('Daily Quantity Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Price over time for key products to visualize promotions / price changes

key_products = [
    'Breakfast Cereal 500g',
    'Cola 6-pack',
    'Whole Milk 1L',
]

price_ts = (df[df['product_name'].isin(key_products)]
              .groupby(['date', 'product_name'])['unit_price']
              .mean()
              .reset_index())

plt.figure(figsize=(12, 6))
for p in key_products:
    subset = price_ts[price_ts['product_name'] == p]
    plt.plot(subset['date'], subset['unit_price'], marker='o', label=p)

plt.legend()
plt.ylabel('Average Unit Price')
plt.title('Unit Price Over Time for Selected Products')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Category-level revenue share

cat_summary = df.groupby('category')['line_total'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
cat_summary.plot(kind='bar')
plt.ylabel('Revenue')
plt.title('Revenue by Category')
plt.tight_layout()
plt.show()